<a href="https://colab.research.google.com/github/MGentieu/Challenge_welding/blob/martin_branch_integration/Example_solution%20Phil.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Assurez-vous de bien clôner le projet dans le terminal dans le dossier /content :

git clone https://github.com/MGentieu/Challenge_welding.git

### Lien vers le projet : Pointe vers le dossier principal

In [1]:
import os
import sys
from pathlib import Path

In [2]:
# The user has provided the explicit path to the project root.
PROJECT_ROOT = Path("/content/Challenge_welding")

print(f"Environment: Colab/Kaggle (remote server), using provided PROJECT_ROOT")

# Validate structure
if not (PROJECT_ROOT / "challenge_solution").exists():
    raise FileNotFoundError(f"Missing src/ directory at {PROJECT_ROOT}")

# Setup Python path
os.chdir(PROJECT_ROOT)
src_path = str(PROJECT_ROOT / "challenge_solution")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"Project root: {PROJECT_ROOT}")
print(f"Working directory: {Path.cwd()}")


Environment: Colab/Kaggle (remote server), using provided PROJECT_ROOT
Project root: /content/Challenge_welding
Working directory: /content/Challenge_welding


In [3]:
import numpy as np
import cv2
import time

import torch
from torchvision import transforms
from torchvision.transforms import InterpolationMode
#sys.path.insert(0, '/home/kevin.pasini/projet_explo/kevin/uqmodels/abench/')
import challenge_solution.df_utils as dm
from challenge_solution.torch_dataloader import ImageDataFrameDataset
from challenge_solution.AIComponent import MyAIComponent

### On vérifie si cuda est disponible :

In [4]:
myDevice = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {myDevice}")

Using device: cuda


On monte ensuite drive qui contient le dataset

In [5]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#%load_ext autoreload
#%autoreload 2

In [6]:


# Exemple de transform basique Redimensionne et normalise
transform = transforms.Compose([transforms.Resize(size=(224, 224), interpolation=InterpolationMode.BILINEAR, max_size=None, antialias=True),
                                transforms.ToTensor(),
                                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

#Genere un meta dataframe utilisé pour acceder au données.
dossier_des_images = "/content/drive/MyDrive/data_challenge_welding/welding-detection-challenge-dataset/"
df_data = dm.explore_csv_hierarchy(dossier_des_images,depth_name_list=['seam','decision','type_label'],allowed_ext='.jpeg')
df_data['path'] = df_data['path'].apply(lambda p: os.path.relpath(p, dossier_des_images))

# Mapping des labels
mapping = {'OK': 0, 'KO': 1}
df_data['label'] = df_data['decision'].map(mapping)

print("Types de décisions :", df_data['decision'].unique())
print("Valeurs manquantes :", df_data['label'].isna().sum())
print("Types de soudure :", df_data['type_label'].unique())

welding_types = ["c20", "c33", "c102"]

for wt in welding_types:
    print(f"\n=== Entraînement modèle pour {wt} ===")

    # Filtrer le dataset sur le type de soudure (seam)
    df_subset = df_data[df_data['seam'] == wt]

    if df_subset.empty:
        print(f"⚠️  Aucun échantillon trouvé pour {wt}, vérifie tes chemins et extensions.")
        continue

    # Stratified split
    df_train, df_val = dm.stratified_train_val_split(df_subset, ['seam','decision'], alpha=0.95, random_state=42)

    # Créer les datasets
    Train_Dataset = ImageDataFrameDataset(
        df=df_train,
        root_dir="/content/drive/MyDrive/data_challenge_welding/welding-detection-challenge-dataset/",
        #root_dir="./datasets/welding-detection-challenge-dataset/",
        path_col="path",
        label_col="label",
        channels_first=True,
        is_train=True,
        welding_mode="balanced"
    )

    Val_Dataset = ImageDataFrameDataset(
        df=df_val,
        root_dir="/content/drive/MyDrive/data_challenge_welding/welding-detection-challenge-dataset/",
        #root_dir="./datasets/welding-detection-challenge-dataset/",
        path_col="path",
        label_col="label",
        channels_first=True,
        is_train=False
    )

    # Initialiser et entraîner
    ai_component = MyAIComponent()
    ai_component.init_model()
    ai_component.train_model(
        Train_Dataset,
        Val_Dataset,
        device=myDevice,  # 'cpu' ou 'cuda' si disponible
        save_path=f"best_model_{wt}.pth",
        augmentation_fn=None,
        preprocess_fn=None,
        epochs=1,
        batch_size=64,
        lr=3e-4
    )


print("\n🎉 Entraînement des trois modèles terminé !")


Types de décisions : ['KO' 'OK']
Valeurs manquantes : 0
Types de soudure : ['operator' 'expert']

=== Entraînement modèle pour c20 ===
🟦 Training started...


Training:   0%|          | 0/219 [01:28<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# --- Initialisation du composant AI ---
ai_component = MyAIComponent()

# --- Fonction utilitaire pour charger et convertir une image en RGB ---
def load_image(image_path: str) -> np.ndarray:
    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(f"Image introuvable : {image_path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# --- Fonction de prédiction pour une image et son type de soudure ---
def predict_welding_image(image_path: str, welding_type: str):
    # Charger l'image
    img = load_image(image_path)

    # Charger le modèle spécialisé correspondant
    ai_component.load_model(welding_type)  # <-- IMPORTANT !

    # Préparer les métadonnées
    metadata = [{"type_label": welding_type}]

    # Faire la prédiction via le modèle spécialisé
    results = ai_component.predict([img], metadata)

    # Extraire résultats
    pred = results['predictions'][0]
    probs = results['probabilities'][0]
    ood = results['OOD_scores'][0]

    # Affichage clair
    print(f"\n🔹 Image : {image_path}")
    print(f"  Modèle utilisé : {welding_type}")
    print(f"  Prediction : {pred}")
    print(f"  Probabilités : OK={probs[0]:.3f}, KO={probs[1]:.3f}, UNKNOWN={probs[2]:.3f}")
    print(f"  OOD score : {ood:.3f}")

# --- Exemple d'utilisation ---
#image_path = "./datasets/welding-detection-challenge-dataset/c20/KO/expert/sample_357.jpeg"
image_path = "./datasets/welding-detection-challenge-dataset/c33/OK/expert/sample_10.jpeg"


predict_welding_image(image_path, welding_type="c33")  # c20 / c33 / c102 selon le type de soudure
